In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch


dataset = load_dataset(
    "dim/hendrycks_math_train_1k_DeepSeek-R1-Distill-Qwen-1.5B_max_len_4096_greedy"
)
dataset = dataset["train"].train_test_split(
    # test_size=250,
    test_size=350,
    # test_size=999,
    # test_size=1,
    seed=42,
)
dataset = dataset["test"].filter(lambda x: x["model_answer"].count("</think>") == 1)

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    # attn_implementation="sdpa",
    attn_implementation="flash_attention_2",
)
model.requires_grad_(False)
tokenizer = AutoTokenizer.from_pretrained(model_name)

### Кодируем части текста в вектора

In [2]:
!pip install more_itertools -q

### Fixed train part

In [2]:
from more_itertools import chunked
from tqdm import tqdm
import itertools
import torch
import math

start_item = 100
cramming_tokens = []

for i in range(start_item, len(dataset)):
    print(f"{i}/{len(dataset)}")
    model_answer = dataset[i]["model_answer"]
    tokens = tokenizer.encode(
        dataset[i]["model_answer"],
        add_special_tokens=False,
    )
    mem_tokens = 8
    encode_size = mem_tokens * 4
    tokens_chunks = list(chunked(tokens, encode_size))
    max_parts = 10
    for chunk_part in range(2, max_parts):
        train_part = tokens_chunks[:chunk_part]
        train_part = list(itertools.chain(*train_part))

        train_part = torch.tensor(
            train_part,
            device="cuda",
        ).unsqueeze(0)

        target_len = encode_size + 1
        num_repeats = (target_len + mem_tokens - 1) // mem_tokens
        new_compression_len = num_repeats * mem_tokens

        context_len = train_part.shape[1] - encode_size
        if context_len < 0:
            continue

        labels_part = torch.full(
            (1, context_len + new_compression_len),
            -100,
            device="cuda",
            dtype=torch.long,
        )

        original_target_tokens = train_part[:, -encode_size:]

        new_target_labels = torch.cat(
            [
                original_target_tokens[:, 0].unsqueeze(1),
                original_target_tokens,
            ],
            dim=1,
        )

        labels_part[:, context_len : context_len + target_len] = new_target_labels

        compression_tensor_param = torch.nn.Parameter(
            torch.rand(
                mem_tokens,
                model.get_input_embeddings().weight.shape[1],
                device="cuda",
            ).unsqueeze(0),
            requires_grad=True,
        )
        optimizer = torch.optim.AdamW(
            [compression_tensor_param],
            lr=0.1,
        )

        prev_tokens = train_part[:, :-encode_size].clone()
        prev_embeds = model.get_input_embeddings()(prev_tokens)

        epoch_amount = 50
        dtype = torch.bfloat16
        for epoch in tqdm(range(epoch_amount)):
            compression_tensor = compression_tensor_param.repeat(1, num_repeats, 1)

            input_embeds = torch.cat(
                [
                    prev_embeds,
                    compression_tensor,
                ],
                dim=1,
            ).to(dtype)

            assert input_embeds.shape[1] == labels_part.shape[1]

            model_predicts = model(
                inputs_embeds=input_embeds,
                labels=labels_part,
            )

            compression_loss = model_predicts.loss
            compression_loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        # --- НАЧАЛО ИЗМЕНЕНИЙ В ЛОГИКЕ ПРОВЕРКИ ---

        # 1. Эталонная последовательность для проверки - это все метки, КРОМЕ ПОСЛЕДНЕЙ.
        #    Ее длина `encode_size`.
        check_target_labels = new_target_labels[0, 1:]
        check_target_str = tokenizer.decode(
            check_target_labels, add_special_tokens=False
        )

        # 2. Извлекаем предсказания модели. Нам нужна та же длина, что и у эталона.
        predicted_tokens = model_predicts.logits.argmax(-1)
        # Берем срез длиной `target_len`, как и раньше...
        predicted_target_part = predicted_tokens[
            :, context_len : context_len + target_len
        ][0]

        # 3. ...но для сравнения отбрасываем последний токен.
        prediction_str = tokenizer.decode(
            predicted_target_part[:-1], add_special_tokens=False
        )

        # 4. Сравниваем строки длиной `encode_size`.
        correct_reconstruction = prediction_str == check_target_str

        # --- КОНЕЦ ИЗМЕНЕНИЙ В ЛОГИКЕ ПРОВЕРКИ ---
        print(f"FULL TEXT: '{tokenizer.decode(train_part[:, :][-1])}'")
        print("-" * 50)
        print(
            f"Context: '{tokenizer.decode(train_part[:, :-encode_size][-1])}'",
        )
        # Эталонная строка, которую модель должна была сгенерировать
        print(
            f"TARGET to generate: '{check_target_str}'",
        )
        # Строка, которую модель сгенерировала на самом деле
        print(f"PREDICTED string:   '{prediction_str}'")
        print(f"Correct reconstruction: '{correct_reconstruction}'")
        print("=" * 50)
        print("=" * 50)
        cramming_tokens.append(compression_tensor_param.detach())
        # break

    break

100/209


  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:01<00:00, 31.83it/s]


FULL TEXT: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32,'
--------------------------------------------------
Context: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I'
TARGET to generate: ' remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32,'
PREDICTED string:   ' remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32,'
Correct reconstruction: 'True'


100%|██████████| 50/50 [00:01<00:00, 38.11it/s]


...

FULL TEXT: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer. 

Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1=32, 32×2=64, 32×3=96, and so on. So, the first one is'
--------------------------------------------------
Context: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me thin

100%|██████████| 50/50 [00:01<00:00, 34.45it/s]


FULL TEXT: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer. 

Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1=32, 32×2=64, 32×3=96, and so on. So, the first one is 32 itself. Therefore, 32 is indeed the smallest positive multiple of 32.

But wait, is there a trick here? Maybe the'
--------------------------------------

100%|██████████| 50/50 [00:01<00:00, 33.50it/s]

FULL TEXT: 'Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer. 

Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1=32, 32×2=64, 32×3=96, and so on. So, the first one is 32 itself. Therefore, 32 is indeed the smallest positive multiple of 32.

But wait, is there a trick here? Maybe the problem is trying to trick me into thin

### Decode part

In [3]:
import torch
from more_itertools import chunked
import itertools

# --- ПРЕДПОЛАГАЕТСЯ, ЧТО ЭТИ ОБЪЕКТЫ УЖЕ СУЩЕСТВУЮТ В ВАШЕЙ СРЕДЕ ---
# model: ваша языковая модель, уже загруженная на GPU
# tokenizer: ваш токенизатор
# dataset: ваш датасет
# cramming_tokens: список с обученными тензорами
# --------------------------------------------------------------------------


def predict_in_one_pass(
    model,
    tokenizer,
    context_tokens: torch.Tensor,
    learned_compression_tensor: torch.Tensor,
    expected_len: int,
):
    """
    Выполняет один прямой проход и извлекает предсказанные токены.
    Возвращает декодированный текст и список ID токенов.
    (Эта функция остается без изменений)
    """
    model.eval()
    device = model.device
    dtype = model.dtype

    context_tokens = context_tokens.to(device)
    mem_tokens = learned_compression_tensor.shape[1]
    target_len = expected_len + 1
    num_repeats = (target_len + mem_tokens - 1) // mem_tokens
    repeated_compression_tensor = learned_compression_tensor.repeat(
        1, num_repeats, 1
    ).to(dtype)

    with torch.no_grad():
        context_embeds = model.get_input_embeddings()(context_tokens)
        input_embeds = torch.cat(
            [context_embeds, repeated_compression_tensor],
            dim=1,
        )
        outputs = model(inputs_embeds=input_embeds)
        context_len = context_embeds.shape[1]
        predicted_logits = outputs.logits[
            :, context_len : context_len + expected_len, :
        ]
        predicted_token_ids = torch.argmax(predicted_logits, dim=-1)
        reconstructed_text = tokenizer.decode(
            predicted_token_ids[0], skip_special_tokens=True
        )

    return reconstructed_text, predicted_token_ids[0].tolist()


# --- ОСНОВНАЯ ЛОГИКА С ПОЛНЫМ ВЫВОДОМ И ОСТАНОВКОЙ ПРИ ОШИБКЕ ---

# Параметры
start_item = 100
mem_tokens = 8
encode_size = mem_tokens * 4
num_chunks_to_test = 5

print(
    f"Запускаем восстановление для {num_chunks_to_test} чанков С ОСТАНОВКОЙ ПРИ ПЕРВОЙ ОШИБКЕ..."
)
print("=" * 80)

# 1. Получаем исходные данные
item_index = start_item
model_answer = dataset[item_index]["model_answer"]
tokens = tokenizer.encode(model_answer, add_special_tokens=False)
tokens_chunks = list(chunked(tokens, encode_size))

# 2. Проверка данных
if len(tokens_chunks) < num_chunks_to_test + 1:
    raise ValueError(
        f"Недостаточно чанков в тексте ({len(tokens_chunks)}) для теста {num_chunks_to_test} чанков."
    )
if len(cramming_tokens) < num_chunks_to_test:
    raise ValueError(
        f"Недостаточно обученных тензоров ({len(cramming_tokens)}) для теста {num_chunks_to_test} чанков."
    )

# --- ДОБАВЛЕНО ДЛЯ ОТЛАДКИ: Вывод полного эталонного текста ---
total_chunks_to_process = 1 + num_chunks_to_test
original_tokens_goal = list(itertools.chain(*tokens_chunks[:total_chunks_to_process]))
print("--- Полный эталонный текст для восстановления ---")
print(tokenizer.decode(original_tokens_goal))
print("=" * 80)
# ---------------------------------------------------------------

# 3. Начинаем цикл восстановления
current_context_tokens = list(tokens_chunks[0])
total_correct_chunks = 0
last_processed_chunk_index = -1

for i in range(num_chunks_to_test):
    last_processed_chunk_index = i
    print(f"\n--- Восстановление чанка {i+1}/{num_chunks_to_test} ---")

    # a. Подготовка данных
    context_tensor = torch.tensor([current_context_tokens], dtype=torch.long)
    learned_tensor = cramming_tokens[i]
    original_target_chunk = tokens_chunks[i + 1]
    original_target_text = tokenizer.decode(original_target_chunk)

    # --- ИЗМЕНЕНО ДЛЯ ОТЛАДКИ: Полный вывод контекста ---
    print(
        f"Текущий контекст (полный восстановленный текст, {len(current_context_tokens)} токенов):"
    )
    print(f"'{tokenizer.decode(current_context_tokens)}'")
    print("-" * 40)
    # ----------------------------------------------------

    # b. Инференс
    reconstructed_text, reconstructed_token_ids = predict_in_one_pass(
        model=model,
        tokenizer=tokenizer,
        context_tokens=context_tensor,
        learned_compression_tensor=learned_tensor,
        expected_len=encode_size,
    )

    # c. Вывод результатов шага
    print(f"ЭТАЛОН для этого шага:    '{original_target_text}'")
    print(f"РЕЗУЛЬТАТ этого шага: '{reconstructed_text}'")

    # d. Проверка и обновление контекста или остановка цикла
    if original_target_text.strip() == reconstructed_text.strip():
        print("✅ Результат верный.")
        total_correct_chunks += 1
        current_context_tokens.extend(reconstructed_token_ids)
    else:
        print("❌ ОШИБКА в восстановлении.")
        print(
            "Останавливаю дальнейшее восстановление, так как контекст был бы искажен."
        )
        current_context_tokens.extend(reconstructed_token_ids)
        break

# --- Итоговые результаты ---
print("\n" + "=" * 80)
print("             ИТОГОВЫЕ РЕЗУЛЬТАТЫ")
print("=" * 80)
print(
    f"Успешно восстановлено: {total_correct_chunks} из {last_processed_chunk_index + 1} предпринятых попыток."
)

# Собираем эталонный текст той же длины, что и восстановленный, для прямого сравнения
# Он будет состоять из 1 начального чанка и `last_processed_chunk_index + 1` целевых чанков
num_original_chunks_to_compare = 1 + last_processed_chunk_index + 1
original_tokens_to_compare = list(
    itertools.chain(*tokens_chunks[:num_original_chunks_to_compare])
)
original_full_text = tokenizer.decode(original_tokens_to_compare)

reconstructed_full_text = tokenizer.decode(current_context_tokens)

print("\n--- Оригинальный текст (до точки остановки) ---")
print(original_full_text)
print("\n--- Восстановленный текст (до точки остановки) ---")
print(reconstructed_full_text)
print("=" * 80)

Запускаем восстановление для 5 чанков С ОСТАНОВКОЙ ПРИ ПЕРВОЙ ОШИБКЕ...
--- Полный эталонный текст для восстановления ---
Okay, so I have this problem: "What is the smallest positive multiple of 32?" Hmm, let me think about how to approach this. I remember that a multiple of a number is just that number multiplied by an integer. So, if I'm looking for the smallest positive multiple of 32, I need to find the smallest positive integer that, when multiplied by 32, gives me another positive integer. 

Wait, but isn't every number a multiple of itself? Like, 32 times 1 is 32, which is a multiple of 32. So, isn't 32 the smallest positive multiple? That seems too straightforward. Maybe I'm missing something here.

Let me double-check. The definition of a multiple is a number that can be expressed as n times another integer, where n is a positive integer. So, for 32, the multiples would be 32×1

--- Восстановление чанка 1/5 ---
Текущий контекст (полный восстановленный текст, 32 токенов):
'Okay